# LeetCode #1080: Insufficient Nodes in Root to Leaf Paths

https://leetcode.com/problems/insufficient-nodes-in-root-to-leaf-paths/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: DFS Post-Order Pruning ★** | $O(n)$ | $O(h)$ |

---

## Understanding the Methods

### Brute Force
Enumerate all root-to-leaf paths, compute their sums, then mark and delete nodes that don't appear on any path with sum $\geq \text{limit}$. Multiple passes make this quadratic.

### Optimal: DFS Post-Order Pruning ★
A single post-order DFS subtracts each node's value from `limit` as it descends. At a leaf, if the remaining limit is positive the leaf is insufficient and is pruned (return null). An internal node is pruned only when both children have been pruned — meaning no path through it satisfies the limit.

**Constraints:**
* The number of nodes is in $[1, 5000]$
* $-10^5 \leq \text{Node.val} \leq 10^5$
* $-10^9 \leq \text{limit} \leq 10^9$

## Solutions

### C#

In [ ]:
public class Solution {
    public TreeNode SufficientSubset(TreeNode root, int limit) {
        // Subtract this node's contribution and prune if the path can't reach limit
        if (root == null) return null;
        limit -= root.val;
        // Leaf: prune if no path through it meets the threshold
        if (root.left == null && root.right == null)
            return limit <= 0 ? root : null;
        root.left  = SufficientSubset(root.left,  limit);
        root.right = SufficientSubset(root.right, limit);
        // Internal node survives only if at least one child survived
        return (root.left == null && root.right == null) ? null : root;
    }
}

### Python

In [ ]:
class Solution:
    def sufficient_subset(self, root, limit: int):
        # Subtract this node's contribution and prune if the path can't reach limit
        if root is None:
            return None
        limit -= root.val
        # Leaf: prune if no path through it meets the threshold
        if root.left is None and root.right is None:
            return None if limit > 0 else root
        root.left  = self.sufficient_subset(root.left,  limit)
        root.right = self.sufficient_subset(root.right, limit)
        # Internal node survives only if at least one child survived
        return root if root.left or root.right else None

### Go

In [ ]:
func sufficientSubset(root *TreeNode, limit int) *TreeNode {
    // Subtract this node's contribution and prune if the path can't reach limit
    if root == nil { return nil }
    limit -= root.Val
    // Leaf: prune if no path through it meets the threshold
    if root.Left == nil && root.Right == nil {
        if limit > 0 { return nil }
        return root
    }
    root.Left  = sufficientSubset(root.Left,  limit)
    root.Right = sufficientSubset(root.Right, limit)
    // Internal node survives only if at least one child survived
    if root.Left == nil && root.Right == nil { return nil }
    return root
}

### Rust

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;
impl Solution {
    pub fn sufficient_subset(root: Option<Rc<RefCell<TreeNode>>>, limit: i32) -> Option<Rc<RefCell<TreeNode>>> {
        if let Some(node) = root {
            let val = node.borrow().val;
            let rem = limit - val;
            let left  = node.borrow().left.clone();
            let right = node.borrow().right.clone();
            // Leaf: prune if no path through it meets the threshold
            if left.is_none() && right.is_none() {
                return if rem > 0 { None } else { Some(node) };
            }
            // Subtract this node's contribution and prune if the path can't reach limit
            let new_left  = Self::sufficient_subset(left,  rem);
            let new_right = Self::sufficient_subset(right, rem);
            node.borrow_mut().left  = new_left.clone();
            node.borrow_mut().right = new_right.clone();
            // Internal node survives only if at least one child survived
            if new_left.is_none() && new_right.is_none() { None } else { Some(node) }
        } else { None }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `root = [1,2,3,4,-99,-99,7,8,1,-99,2,null,null,null,null]`, `limit = 1`
Leaves with insufficient path sums are pruned in a post-order pass. The nodes $-99$ drag several paths below the limit, and their parent nodes are pruned if they have no surviving child.

### 2. Slightly Complex
**Input:** `root = [5,4,8,11,null,17,4,7,1,null,null,null,5]`, `limit = 22`
Path $5 \to 4 \to 11 \to 7 = 27 \geq 22$ survives; $5 \to 4 \to 11 \to 1 = 21 < 22$ prunes the leaf $1$. The post-order pass removes only what cannot contribute to any valid path.

### 3. Edge Case: Time Factor
**Input:** A linear chain (right-skewed tree) of 5000 nodes, all values $1$, `limit = 5001`
The chain has exactly one root-to-leaf path with sum $5000 < 5001$. The DFS descends the full depth before any pruning decision is made — maximum call stack usage before the leaf triggers removal of the entire chain.

### 4. Edge Case: Space Factor
**Input:** Single node with `val = 5`, `limit = 5`
`limit - 5 = 0 \leq 0` at the leaf; the node survives. Call stack depth is 1. Minimum possible execution.

### 5. Almost-Impossible but Plausible
**Input:** Root value $-10^5$, `limit = -10^9$`
After subtracting the root, remaining limit is $-10^9 + 10^5$, still deeply negative. Any path reaches the leaf with a negative remaining limit, so all leaves survive. Confirms the algorithm handles large negative values without integer overflow when using $-10^9 \leq \text{limit} \leq 10^9$ and $-10^5 \leq \text{val}$.